[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/ml/04-machine-learning/ml-multilabel.ipynb)

# Multi-Label & Multi-Output Learning

*AIBits Academy · Machine Learning End To End · Advanced Extra Topic · New*

Every model so far predicted exactly one target per sample. Real problems are often messier — a product belongs to several categories at once, or you need to predict several related numbers together.

**How to use this notebook:** run the cells top to bottom (Runtime → Run all). Each code cell is the same code you saw on the course page, so you can compare your output with the lesson. The graded exercises are at the end; try them before opening the solutions.

*Interactive animations and quiz cards stay on the course page.*

*Small numeric differences from the lesson page are normal: library versions, random seeds and dataset copies change the last digits. The conclusions should agree.*

## Four Prediction Problem Shapes, Not Two

| Problem type | y shape | Example |
|---|---|---|
| Single-label classification | One class out of many, mutually exclusive | A restaurant review is Positive OR Negative OR Neutral |
| **Multi-label classification** | Any subset of several labels, not mutually exclusive | A Flipkart product can be tagged "Electronics" AND "Gifts" AND "Bestseller" simultaneously |
| Single-output regression | One continuous number | Predict an apartment's price |
| **Multi-output regression** | Several continuous numbers together | Predict an apartment's price AND expected days-on-market together |

Everything on this page is genuinely different from multi*class* classification (covered back on the Logistic Regression page via Softmax) — multiclass still picks exactly **one** class per sample from several options; multi-**label** can pick **any number** of labels per sample, including zero or all of them.

## Multi-Label Classification — Code

In [ ]:
import numpy as np
from sklearn.multioutput import MultiOutputClassifier, ClassifierChain
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import hamming_loss, accuracy_score

# Flipkart product tagging: 3 independent binary tags per product
# [is_electronics, is_gift_suitable, is_bestseller] — any combination is valid
np.random.seed(21)
n = 1000
price = np.random.uniform(100,50000,n)
rating = np.random.uniform(2,5,n)
category_score = np.random.uniform(0,1,n)
X = np.column_stack([price, rating, category_score])

is_electronics = (category_score > 0.6).astype(int)
is_gift = ((price < 3000) & (rating > 3.5)).astype(int)
is_bestseller = ((rating > 4.2) & (np.random.rand(n) > 0.5)).astype(int)
Y = np.column_stack([is_electronics, is_gift, is_bestseller])  # (n_samples, 3) — NOT a single column

X_tr, X_te, Y_tr, Y_te = train_test_split(X, Y, test_size=0.2, random_state=42)

# Approach 1: MultiOutputClassifier — trains one independent classifier per label
moc = MultiOutputClassifier(RandomForestClassifier(n_estimators=100, random_state=42))
moc.fit(X_tr, Y_tr)
preds = moc.predict(X_te)

print(f"Hamming loss (fraction of individual label mistakes): {hamming_loss(Y_te, preds):.3f}")
print(f"Exact-match (subset) accuracy — ALL 3 labels correct: {accuracy_score(Y_te, preds):.3f}")
# Exact-match is a much stricter, often more informative metric than treating each label separately

## Why Label Correlations Matter — ClassifierChain

`MultiOutputClassifier` trains each label's classifier completely independently, ignoring any relationship between labels — but in reality, "is_bestseller" is far more likely given "is_electronics=1", not independent of it. `ClassifierChain` instead feeds each label's prediction as an *additional input feature* to the next label's classifier, in a chosen order, explicitly modelling label dependencies:

$$P(y_1,y_2,y_3\mid x) = P(y_1\mid x)\cdot P(y_2\mid x,y_1)\cdot P(y_3\mid x,y_1,y_2) \qquad \text{(chain rule of probability, applied to labels)}$$

In [ ]:
chain = ClassifierChain(RandomForestClassifier(n_estimators=100, random_state=42), order='random', random_state=1)
chain.fit(X_tr, Y_tr)
chain_preds = (chain.predict(X_te) > 0.5).astype(int)
print(f"ClassifierChain exact-match accuracy: {accuracy_score(Y_te, chain_preds):.3f}")
# Chain order matters and is itself a tunable hyperparameter — try averaging predictions
# across several random chain orders (an ensemble of chains) for more robust results

## Multi-Output Regression — Predicting Several Numbers Together

In [ ]:
from sklearn.multioutput import MultiOutputRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge  # natively supports multi-output — no wrapper needed

# Ahmedabad apartment listing: predict price AND days-on-market together
np.random.seed(5)
area = np.random.randint(500,2500,500); locality_score = np.random.uniform(0,1,500)
Xr = np.column_stack([area, locality_score])
price_lakh = 0.05*area + 40*locality_score + np.random.normal(0,5,500)
days_on_market = 60 - 30*locality_score + np.random.normal(0,8,500)
Yr = np.column_stack([price_lakh, days_on_market])

# Ridge handles multi-output natively (it's just multiple independent linear fits under the hood)
ridge_multi = Ridge().fit(Xr, Yr)
# Random Forest needs the explicit wrapper
rf_multi = MultiOutputRegressor(RandomForestRegressor(n_estimators=100, random_state=42)).fit(Xr, Yr)

sample = np.array([[1400, 0.7]])
pred_price, pred_days = ridge_multi.predict(sample)[0]
print(f"Predicted: ₹{pred_price:.1f} lakhs, {pred_days:.0f} days on market")

## Evaluation Metrics — What Changes

| Metric | Measures | Use for |
|---|---|---|
| Hamming Loss | Fraction of individual label predictions that are wrong, averaged | Multi-label — lenient, per-label view |
| Subset (Exact-Match) Accuracy | Fraction of samples where *every* label was predicted correctly | Multi-label — strict, all-or-nothing view |
| Jaccard Score | Overlap between predicted and true label sets, per sample, averaged | Multi-label — a middle ground between Hamming and exact-match |
| Per-target R² / RMSE | Standard regression metrics, computed separately per output column | Multi-output regression |

## Try It — Watch Exact-Match Accuracy Collapse as Labels Are Added

Set a per-label accuracy and a number of labels below. The orange curve plots exact-match accuracy = (per-label accuracy)<sup>n</sup> across label counts 1–15, with a marker at your current settings — the same arithmetic behind the Q&A's 0.9³≈73% and 0.9¹&sup0≈35% figures above.

### ❓ Conceptual Q&A

---
## Graded exercises

Each exercise has a **starter cell** you complete and a **check cell** that prints ✅ or ❌. The solution is folded away underneath — try first.

In [ ]:
# --- self-check helper (used by the exercises) ---------------------------------------------
def check(name, ok):
    print(("\u2705 " if ok else "\u274c ") + name)


### Exercise 1 · Easy · Hamming loss by hand

Hamming loss is the fraction of individual label slots predicted wrongly. For the arrays below (rows = products, columns = tags) store it in `hl`.

In [ ]:
import numpy as np
Y_true = np.array([[1, 0, 1], [0, 1, 0], [1, 1, 0], [0, 0, 1]])
Y_pred = np.array([[1, 0, 0], [0, 1, 0], [1, 0, 0], [0, 0, 1]])
hl = None   # TODO


In [ ]:
try:
    from sklearn.metrics import hamming_loss
    check("matches sklearn", abs(hl - hamming_loss(Y_true, Y_pred)) < 1e-12)
    check("value", abs(hl - 2 / 12) < 1e-12)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import numpy as np
Y_true = np.array([[1, 0, 1], [0, 1, 0], [1, 1, 0], [0, 0, 1]])
Y_pred = np.array([[1, 0, 0], [0, 1, 0], [1, 0, 0], [0, 0, 1]])
hl = float((Y_true != Y_pred).mean())

```

</details>

### Exercise 2 · Medium · Cardinality and density

Store the **label cardinality** (average number of tags per product) in `cardinality` and the **label density** (cardinality divided by the number of possible tags) in `density`, for `Y_true` above.

In [ ]:
cardinality = density = None   # TODO


In [ ]:
try:
    check("cardinality", abs(cardinality - 1.5) < 1e-12)
    check("density", abs(density - 0.5) < 1e-12)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
cardinality = float(Y_true.sum(axis=1).mean())
density = cardinality / Y_true.shape[1]

```

</details>

### Exercise 3 · Stretch · Binary relevance from scratch

Multi-label 'binary relevance' trains **one classifier per label**. Build it manually: for each column of `Y`, fit a `LogisticRegression` and stack the predicted columns into `Y_hat` (shape n × 3). It must equal `MultiOutputClassifier(LogisticRegression())`'s predictions.

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.multioutput import MultiOutputClassifier
rng = np.random.default_rng(0)
X = rng.normal(size=(300, 4))
Y = np.column_stack([X[:, 0] > 0, X[:, 1] + X[:, 2] > 0, X[:, 3] > 0.5]).astype(int)
Y_hat = None   # TODO


In [ ]:
try:
    ref = MultiOutputClassifier(LogisticRegression()).fit(X, Y).predict(X)
    check("shape", Y_hat.shape == (300, 3))
    check("same predictions as the wrapper", np.array_equal(Y_hat, ref))
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.multioutput import MultiOutputClassifier
rng = np.random.default_rng(0)
X = rng.normal(size=(300, 4))
Y = np.column_stack([X[:, 0] > 0, X[:, 1] + X[:, 2] > 0, X[:, 3] > 0.5]).astype(int)
Y_hat = np.column_stack([LogisticRegression().fit(X, Y[:, j]).predict(X) for j in range(Y.shape[1])])

```

Binary relevance ignores correlations between labels; classifier chains add them by feeding earlier predictions forward.

</details>

---
*Back to the course: **Machine Learning End To End → Multi-Label & Multi-Output Learning**.*